# Federated Learning Based Nepali Grammar Checking - Fixed Version
## With Flowers (flwr) Framework Integration - UPDATED API

In [10]:
# Install required packages
import subprocess
import sys

packages = ['flwr>=1.8.0', 'torch', 'pandas', 'numpy', 'scikit-learn']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All packages installed!")

✓ All packages installed!


In [11]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import numpy as np
import flwr as fl
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Flowers version: {fl.__version__}")

PyTorch version: 2.12.0+cu130
Flowers version: 1.31.0


## 1. Create Sample Nepali Dataset

In [39]:
# Load detection + correction data from the same CSV
# CSV columns: Right (correct word), Wrong (incorrect word)

DATA_PATH = "/linux-data/projects/ioe_purwanchal_campus_iicquest4.0/ml/data_cleaned/right_wrong.csv"

df_pairs = pd.read_csv(DATA_PATH, encoding="utf-8")
df_pairs.columns = ["correct", "wrong"]
df_pairs = df_pairs.dropna()
df_pairs = df_pairs[df_pairs["correct"] != df_pairs["wrong"]].reset_index(drop=True)

print(f"Correction pairs loaded: {len(df_pairs)}")
print(df_pairs.head())

# Build detection dataset: correct words -> label 0, wrong words -> label 1
df_correct = pd.DataFrame({"word": df_pairs["correct"].values, "label": 0})
df_wrong   = pd.DataFrame({"word": df_pairs["wrong"].values,   "label": 1})
df = pd.concat([df_correct, df_wrong], ignore_index=True).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print(f"\nDetection dataset shape: {df.shape}")
print(df["label"].value_counts())
print(df.head())

df = df.sample(n=100000, random_state=42).reset_index(drop=True)
print("\nSample data after reduction:")
print(df.shape)


Correction pairs loaded: 2245506
      correct       wrong
0        यसरी        ीसरय
1  व्यवस्थापन  ््नवासयथपव
2      गर्दैछ      छैदगर्
3        बिपी        पिबी
4     कोइराला     लाकाोरइ

Detection dataset shape: (4491012, 2)
label
0    2245506
1    2245506
Name: count, dtype: int64
         word  label
0  दुर्घटनामा      0
1       कमजोर      0
2  स्याङ्जामा      0
3      तगिरएा      1
4    सन्तुष्ट      0

Sample data after reduction:
(100000, 2)


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   word    100000 non-null  object
 1   label   100000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.5+ MB


## 2. Data Preprocessing - FIXED VERSION

In [41]:
# ── Character-level tokenizer for detection ──────────────────────────────
# Word-level tokenizer fails because:
#   - wrong words are OOV -> all map to <UNK>, no signal
#   - vocab size 58k with 100k samples = undertrained embeddings
# Char-level: scrambled chars vs correct chars = learnable pattern

class CharTokenizer:
    PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"

    def __init__(self):
        self.char2idx  = {}
        self.idx2char  = {}
        self.vocab_size = 0

    def build_vocab(self, texts):
        chars = set()
        for t in texts:
            chars.update(list(str(t)))
        specials  = [self.PAD, self.SOS, self.EOS, self.UNK]
        all_chars = specials + sorted(chars)
        self.char2idx   = {c: i for i, c in enumerate(all_chars)}
        self.idx2char   = {i: c for c, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)
        print(f"Char vocab size: {self.vocab_size}")

    def encode(self, text, max_len=30, add_sos=False, add_eos=False):
        ids = []
        if add_sos:
            ids.append(self.char2idx[self.SOS])
        for c in str(text):
            ids.append(self.char2idx.get(c, self.char2idx[self.UNK]))
        if add_eos:
            ids.append(self.char2idx[self.EOS])
        ids = ids[:max_len]
        ids += [self.char2idx[self.PAD]] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        out = []
        for i in ids:
            c = self.idx2char.get(i, self.UNK)
            if c in (self.PAD, self.SOS):
                continue
            if c == self.EOS:
                break
            out.append(c)
        return "".join(out)


MAX_SEQ_LEN = 30   # max chars per word (Nepali words rarely exceed 20 chars)

# Build char vocab from ALL words (correct + wrong)
tokenizer = CharTokenizer()
tokenizer.build_vocab(df["word"].tolist())

# Encode — every word becomes a padded sequence of char indices
X = np.array(
    [tokenizer.encode(w, MAX_SEQ_LEN) for w in df["word"]],
    dtype=np.int64
)
y = df["label"].values

print(f"Encoded shape: {X.shape}")
print(f"Labels shape:  {y.shape}")
print(f"Sample encoding of 'यसरी': {tokenizer.encode('यसरी', MAX_SEQ_LEN)[:10]}...")
print(f"Sample encoding of 'ीसरय': {tokenizer.encode('ीसरय', MAX_SEQ_LEN)[:10]}...")


Char vocab size: 74
Encoded shape: (100000, 30)
Labels shape:  (100000,)
Sample encoding of 'यसरी': [43, 49, 44, 53, 0, 0, 0, 0, 0, 0]...
Sample encoding of 'ीसरय': [53, 49, 44, 43, 0, 0, 0, 0, 0, 0]...


## 3. Split into Train/Test

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_test  = torch.tensor(X_test,  dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

print(f"X_train: {X_train.shape}  dtype: {X_train.dtype}")
print(f"y_train: {y_train.shape}  dtype: {y_train.dtype}")
print(f"X_test:  {X_test.shape}   dtype: {X_test.dtype}")
print(f"y_test:  {y_test.shape}   dtype: {y_test.dtype}")


X_train: torch.Size([75000, 30])  dtype: torch.int64
y_train: torch.Size([75000])  dtype: torch.float32
X_test:  torch.Size([25000, 30])   dtype: torch.int64
y_test:  torch.Size([25000])   dtype: torch.float32


## 4. Detection Model — Transformer Encoder

In [43]:
import math

class CharTransformerDetector(nn.Module):
    """
    Pure Transformer encoder for char-level wrong-word detection.
    Better than BiLSTM for this task:
      - Positional encoding captures char order
      - Self-attention sees all char pairs at once (scrambled vs correct)
      - Faster to train, fewer params
    """
    def __init__(self, vocab_size, embed_dim=64, num_heads=4,
                 num_layers=3, ff_dim=256, max_len=30, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Learned positional encoding
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True        # Pre-LN: more stable training
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.dropout     = nn.Dropout(dropout)
        self.classifier  = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        """
        x: (B, seq_len) char indices
        returns: (B,) probability of being CORRECT (0=wrong, 1=correct)
        """
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)

        # Embed chars + positions
        out = self.dropout(self.embedding(x) + self.pos_embedding(positions))

        # Padding mask: True where PAD (idx=0) so transformer ignores them
        pad_mask = (x == 0)

        # Transformer encoder
        out = self.transformer(out, src_key_padding_mask=pad_mask)  # (B, T, E)

        # Mean pool over non-padding positions
        mask_f  = (~pad_mask).float().unsqueeze(-1)                 # (B, T, 1)
        pooled  = (out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)  # (B, E)

        return self.classifier(pooled).squeeze(-1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

detector = CharTransformerDetector(
    vocab_size = tokenizer.vocab_size,
    embed_dim  = 64,
    num_heads  = 4,
    num_layers = 3,
    ff_dim     = 256,
    max_len    = MAX_SEQ_LEN,
    dropout    = 0.3
).to(device)

print(f"Detector on {device}")
print(f"Parameters: {sum(p.numel() for p in detector.parameters()):,}")
print("Architecture: CharEmbed+PosEmbed -> TransformerEncoder(3L,4H) -> MeanPool -> FC -> Sigmoid")


Detector on cuda
Parameters: 160,833
Architecture: CharEmbed+PosEmbed -> TransformerEncoder(3L,4H) -> MeanPool -> FC -> Sigmoid


## 5. Train Detection Model

In [44]:
def train_detector(model, X_train, y_train, X_test, y_test,
                   epochs=10, batch_size=64):
    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True, num_workers=0
    )
    test_loader  = DataLoader(
        TensorDataset(X_test, y_test),
        batch_size=batch_size, shuffle=False, num_workers=0
    )

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc  = 0.0
    history   = {"train_loss": [], "test_acc": []}

    for epoch in range(epochs):
        # Train
        model.train()
        total_loss = 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        avg_loss = total_loss / len(train_loader)

        # Eval
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for bx, by in test_loader:
                bx, by   = bx.to(device), by.to(device)
                preds    = (model(bx) > 0.5).float()
                correct += (preds == by).sum().item()
                total   += by.size(0)

        acc = correct / total
        history["train_loss"].append(avg_loss)
        history["test_acc"].append(acc)

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), "detector_best.pth")

        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {acc*100:.2f}%"
              + (" <- best" if acc == best_acc else ""))

    print(f"\nBest accuracy: {best_acc*100:.2f}%")
    model.load_state_dict(torch.load("detector_best.pth", map_location=device))
    return history


print("Training Detector...\n")
det_history = train_detector(
    detector, X_train, y_train, X_test, y_test,
    epochs=10, batch_size=64
)
print("\nDetector training complete!")


Training Detector...



Epoch 1/10 | Loss: 0.6023 | Acc: 79.04% <- best
Epoch 2/10 | Loss: 0.4588 | Acc: 83.53% <- best
Epoch 3/10 | Loss: 0.4111 | Acc: 85.91% <- best
Epoch 4/10 | Loss: 0.3815 | Acc: 87.57% <- best
Epoch 5/10 | Loss: 0.3600 | Acc: 88.14% <- best
Epoch 6/10 | Loss: 0.3441 | Acc: 88.70% <- best
Epoch 7/10 | Loss: 0.3331 | Acc: 89.16% <- best
Epoch 8/10 | Loss: 0.3239 | Acc: 89.23% <- best
Epoch 9/10 | Loss: 0.3210 | Acc: 89.22%
Epoch 10/10 | Loss: 0.3210 | Acc: 89.33% <- best

Best accuracy: 89.33%

Detector training complete!


## 6. Evaluate Detection Model

In [45]:
def evaluate_detector(model, X_test, y_test, batch_size=64):
    model.eval()
    loader  = DataLoader(TensorDataset(X_test, y_test),
                         batch_size=batch_size, shuffle=False)
    all_p, all_y = [], []
    with torch.no_grad():
        for bx, by in loader:
            preds = (model(bx.to(device)) > 0.5).float().cpu()
            all_p.append(preds)
            all_y.append(by)

    preds  = torch.cat(all_p)
    labels = torch.cat(all_y)
    acc    = (preds == labels).float().mean().item()
    tp = ((preds==1)&(labels==1)).sum().item()
    fp = ((preds==1)&(labels==0)).sum().item()
    fn = ((preds==0)&(labels==1)).sum().item()
    pre = tp/(tp+fp) if tp+fp else 0
    rec = tp/(tp+fn) if tp+fn else 0
    f1  = 2*pre*rec/(pre+rec) if pre+rec else 0

    print("="*45)
    print("DETECTION EVALUATION")
    print("="*45)
    print(f"Accuracy  : {acc*100:.2f}%")
    print(f"Precision : {pre:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1        : {f1:.4f}")
    print("="*45)
    return {"accuracy": acc, "precision": pre, "recall": rec, "f1": f1}

metrics = evaluate_detector(detector, X_test, y_test)


DETECTION EVALUATION
Accuracy  : 89.33%
Precision : 0.9367
Recall    : 0.8427
F1        : 0.8872


## 7. Seq2Seq Correction Model

In [46]:
# df_pairs already loaded in cell 5 (correct / wrong columns)
print(f"Correction pairs: {len(df_pairs):,}")
print(df_pairs.head())

# Char tokenizer for seq2seq — separate instance, built from full df_pairs vocab
class CharTokenizer:
    PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"

    def __init__(self):
        self.char2idx  = {}
        self.idx2char  = {}
        self.vocab_size = 0

    def build_vocab(self, texts):
        chars = set()
        for t in texts:
            chars.update(list(str(t)))
        specials  = [self.PAD, self.SOS, self.EOS, self.UNK]
        all_chars = specials + sorted(chars)
        self.char2idx   = {c: i for i, c in enumerate(all_chars)}
        self.idx2char   = {i: c for c, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)
        print(f"Seq2Seq char vocab: {self.vocab_size}")

    def encode(self, text, max_len=30, add_sos=False, add_eos=False):
        ids = []
        if add_sos:
            ids.append(self.char2idx[self.SOS])
        for c in str(text):
            ids.append(self.char2idx.get(c, self.char2idx[self.UNK]))
        if add_eos:
            ids.append(self.char2idx[self.EOS])
        ids = ids[:max_len]
        ids += [self.char2idx[self.PAD]] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        out = []
        for i in ids:
            c = self.idx2char.get(i, self.UNK)
            if c in (self.PAD, self.SOS):
                continue
            if c == self.EOS:
                break
            out.append(c)
        return "".join(out)


MAX_WORD_LEN = 30

char_tok = CharTokenizer()
char_tok.build_vocab(df_pairs["correct"].tolist() + df_pairs["wrong"].tolist())

print(f"Encoding {len(df_pairs):,} pairs...")
src_seqs = np.array(
    [char_tok.encode(w, MAX_WORD_LEN) for w in df_pairs["wrong"]], dtype=np.int64
)
tgt_seqs = np.array(
    [char_tok.encode(w, MAX_WORD_LEN, add_sos=True, add_eos=True)
     for w in df_pairs["correct"]], dtype=np.int64
)
print(f"src: {src_seqs.shape}  tgt: {tgt_seqs.shape}")


Correction pairs: 2,245,506
      correct       wrong
0        यसरी        ीसरय
1  व्यवस्थापन  ््नवासयथपव
2      गर्दैछ      छैदगर्
3        बिपी        पिबी
4     कोइराला     लाकाोरइ
Seq2Seq char vocab: 82
Encoding 2,245,506 pairs...
src: (2245506, 30)  tgt: (2245506, 30)


## 8. Seq2Seq Model Architecture

In [47]:
class TransformerEncoder(nn.Module):
    """
    Transformer encoder for seq2seq.
    Better than BiLSTM encoder: parallel, handles long-range char deps better.
    """
    def __init__(self, vocab_size, embed_dim=128, num_heads=4,
                 num_layers=3, ff_dim=512, max_len=30, dropout=0.1):
        super().__init__()
        self.embed_dim   = embed_dim
        self.embedding   = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed   = nn.Embedding(max_len + 2, embed_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.dropout     = nn.Dropout(dropout)
        # Project to decoder hidden size
        self.fc_h        = nn.Linear(embed_dim, embed_dim)
        self.fc_c        = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T     = x.shape
        pos      = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        out      = self.dropout(self.embedding(x) + self.pos_embed(pos))
        pad_mask = (x == 0)
        enc_out  = self.transformer(out, src_key_padding_mask=pad_mask)  # (B,T,E)
        # Build init hidden/cell for decoder from mean of encoder output
        mask_f   = (~pad_mask).float().unsqueeze(-1)
        mean_enc = (enc_out * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)  # (B,E)
        h        = torch.tanh(self.fc_h(mean_enc)).unsqueeze(0)             # (1,B,E)
        c        = torch.tanh(self.fc_c(mean_enc)).unsqueeze(0)
        return enc_out, h, c


class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim, encoder_dim):
        super().__init__()
        self.W1 = nn.Linear(encoder_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim,  hidden_dim)
        self.v  = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, dec_h, enc_out):
        # dec_h: (B, H)  enc_out: (B, T, enc_dim)
        score   = self.v(torch.tanh(
            self.W1(enc_out) + self.W2(dec_h).unsqueeze(1)
        )).squeeze(-1)                                      # (B, T)
        weights = torch.softmax(score, dim=1)               # (B, T)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)  # (B, enc_dim)
        return context, weights


class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, encoder_dim, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = BahdanauAttention(hidden_dim, encoder_dim)
        self.lstm      = nn.LSTMCell(embed_dim + encoder_dim, hidden_dim)
        self.fc_out    = nn.Linear(hidden_dim + encoder_dim + embed_dim, vocab_size)
        self.dropout   = nn.Dropout(dropout)

    def forward_step(self, token, h, c, enc_out):
        emb              = self.dropout(self.embedding(token))   # (B, E)
        context, weights = self.attention(h, enc_out)            # (B, enc_dim)
        h, c             = self.lstm(torch.cat([emb, context], dim=1), (h, c))
        pred             = self.fc_out(torch.cat([h, context, emb], dim=1))
        return pred, h, c, weights


class Seq2SeqCorrector(nn.Module):
    """
    Transformer Encoder + LSTM Decoder + Bahdanau Attention
    Best of both worlds:
      - Transformer encoder: parallel char encoding, better representations
      - LSTM decoder: stable autoregressive generation char by char
      - Bahdanau attention: soft alignment between input/output chars
    """
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 enc_layers=3, dropout=0.1):
        super().__init__()
        self.encoder    = TransformerEncoder(
            vocab_size, embed_dim, num_heads=4,
            num_layers=enc_layers, ff_dim=512,
            max_len=MAX_WORD_LEN, dropout=dropout
        )
        self.decoder    = LSTMDecoder(
            vocab_size, embed_dim, hidden_dim,
            encoder_dim=embed_dim, dropout=dropout
        )
        self.vocab_size = vocab_size

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        B, tgt_len    = tgt.shape
        enc_out, h, c = self.encoder(src)
        h, c          = h.squeeze(0), c.squeeze(0)

        input_tok     = tgt[:, 0]
        outputs       = torch.zeros(B, tgt_len, self.vocab_size).to(src.device)

        for t in range(1, tgt_len):
            pred, h, c, _ = self.decoder.forward_step(input_tok, h, c, enc_out)
            outputs[:, t] = pred
            use_teacher   = torch.rand(1).item() < teacher_forcing_ratio
            input_tok     = tgt[:, t] if use_teacher else pred.argmax(dim=1)

        return outputs


s2s_model = Seq2SeqCorrector(
    vocab_size  = char_tok.vocab_size,
    embed_dim   = 128,
    hidden_dim  = 128,
    enc_layers  = 3,
    dropout     = 0.1
).to(device)

print(f"Seq2Seq model on {device}")
print(f"Parameters: {sum(p.numel() for p in s2s_model.parameters()):,}")
print("Architecture: TransformerEncoder(3L) -> BahdanauAttention -> LSTMDecoder -> char output")


Seq2Seq model on cuda
Parameters: 915,282
Architecture: TransformerEncoder(3L) -> BahdanauAttention -> LSTMDecoder -> char output


## 9. Train Seq2Seq Corrector

In [48]:
from torch.utils.data import random_split

src_t    = torch.tensor(src_seqs, dtype=torch.long)
tgt_t    = torch.tensor(tgt_seqs, dtype=torch.long)
dataset  = TensorDataset(src_t, tgt_t)

train_size = int(0.80 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

BATCH     = 32
train_ldr = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_ldr   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)

PAD_IDX   = char_tok.char2idx["<PAD>"]
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(s2s_model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5
)

EPOCHS       = 20
best_val     = float("inf")

print(f"Train: {train_size:,} | Val: {val_size:,} | Batch: {BATCH} | Epochs: {EPOCHS}\n")

for epoch in range(1, EPOCHS + 1):
    # Train
    s2s_model.train()
    t_loss = 0
    for sb, tb in train_ldr:
        sb, tb = sb.to(device), tb.to(device)
        optimizer.zero_grad()
        out  = s2s_model(sb, tb, teacher_forcing_ratio=0.5)
        loss = criterion(
            out[:, 1:].reshape(-1, char_tok.vocab_size),
            tb[:, 1:].reshape(-1)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(s2s_model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
    avg_t = t_loss / len(train_ldr)

    # Validate
    s2s_model.eval()
    v_loss, correct_w, total_w = 0, 0, 0
    with torch.no_grad():
        for sb, tb in val_ldr:
            sb, tb   = sb.to(device), tb.to(device)
            out      = s2s_model(sb, tb, teacher_forcing_ratio=0.0)
            v_loss  += criterion(
                out[:, 1:].reshape(-1, char_tok.vocab_size),
                tb[:, 1:].reshape(-1)
            ).item()
            pred_ids = out[:, 1:].argmax(dim=-1)
            tgt_ids  = tb[:, 1:]
            mask     = tgt_ids != PAD_IDX
            correct_w += ((pred_ids == tgt_ids) | ~mask).all(dim=1).sum().item()
            total_w   += sb.size(0)

    avg_v    = v_loss / len(val_ldr)
    word_acc = correct_w / total_w * 100
    scheduler.step(avg_v)

    if avg_v < best_val:
        best_val = avg_v
        torch.save(s2s_model.state_dict(), "seq2seq_best.pth")

    print(f"Epoch {epoch:3d}/{EPOCHS} | Train: {avg_t:.4f} | "
          f"Val: {avg_v:.4f} | Word Acc: {word_acc:.1f}%"
          + (" <- best" if avg_v == best_val else ""))

print("\nLoading best checkpoint...")
s2s_model.load_state_dict(torch.load("seq2seq_best.pth", map_location=device))
print("Seq2Seq training complete!")


Train: 1,796,404 | Val: 449,102 | Batch: 32 | Epochs: 20



KeyboardInterrupt: 

## 10. Inference — Beam Search + Top-K Suggestions

In [ ]:
def correct_word_beam(wrong_word, model, tokenizer, max_len=30, beam_width=5):
    """
    Beam search decoding — returns top-3 candidates with confidence scores.
    Each candidate: (corrected_word, score)
    """
    model.eval()
    src = torch.tensor(
        [tokenizer.encode(str(wrong_word), max_len)], dtype=torch.long
    ).to(device)

    SOS = tokenizer.char2idx[tokenizer.SOS]
    EOS = tokenizer.char2idx[tokenizer.EOS]
    PAD = tokenizer.char2idx[tokenizer.PAD]

    with torch.no_grad():
        enc_out, h, c = model.encoder(src)
    h = h.squeeze(0)
    c = c.squeeze(0)

    # (log_prob, token_list, h, c)
    beams     = [(0.0, [], h, c)]
    completed = []

    for _ in range(max_len):
        if not beams:
            break
        candidates = []
        for lp, tokens, bh, bc in beams:
            if tokens and tokens[-1] == EOS:
                completed.append((lp, tokens))
                continue
            last = torch.tensor(
                [tokens[-1] if tokens else SOS], dtype=torch.long
            ).to(device)
            with torch.no_grad():
                pred, new_h, new_c, _ = model.decoder.forward_step(
                    last, bh, bc, enc_out
                )
            log_p = torch.log_softmax(pred[0], dim=-1)
            topk_lp, topk_idx = log_p.topk(beam_width)
            for tlp, tidx in zip(topk_lp.tolist(), topk_idx.tolist()):
                candidates.append((lp + tlp, tokens + [tidx], new_h, new_c))
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]

    # Force-complete any open beams
    for lp, tokens, _, _ in beams:
        completed.append((lp, tokens))

    completed.sort(key=lambda x: x[0], reverse=True)

    def decode_tokens(tokens):
        out = []
        for idx in tokens:
            ch = tokenizer.idx2char.get(idx, tokenizer.UNK)
            if ch == tokenizer.EOS:
                break
            if ch not in (tokenizer.PAD, tokenizer.SOS):
                out.append(ch)
        return "".join(out)

    # Return top-3 unique candidates with normalised confidence
    seen, results = set(), []
    for lp, tokens in completed:
        word = decode_tokens(tokens)
        if word and word not in seen:
            seen.add(word)
            # Normalise score to [0,1] — length-penalised
            norm_score = math.exp(lp / max(len(tokens), 1))
            results.append((word, round(norm_score, 4)))
        if len(results) == 3:
            break

    return results   # [(word, confidence), ...]


def predict_sentence(text, detector, corrector, detect_tok, correct_tok,
                     detect_max=30, correct_max=30, beam_width=5,
                     threshold=0.5):
    """
    Full pipeline:
      1. Split sentence into words
      2. Detector scores each word
      3. Words below threshold get beam-search top-3 suggestions
      4. Best suggestion (rank 1) is used in the output sentence

    Returns dict with corrected sentence + per-word detail.
    """
    detector.eval()
    words   = text.split()
    output  = []
    details = []

    for word in words:
        indices = detect_tok.encode(word, detect_max)
        x       = torch.tensor([indices], dtype=torch.long).to(device)
        with torch.no_grad():
            prob = detector(x).item()

        if prob <= threshold:
            suggestions = correct_word_beam(
                word, corrector, correct_tok, correct_max, beam_width
            )
            best = suggestions[0][0] if suggestions else word
            details.append({
                "word":        word,
                "status":      "wrong",
                "confidence":  round(prob, 4),
                "suggestions": suggestions,   # top-3 [(word, score), ...]
                "corrected":   best
            })
            output.append(best)
        else:
            details.append({
                "word":        word,
                "status":      "correct",
                "confidence":  round(prob, 4),
                "suggestions": [],
                "corrected":   word
            })
            output.append(word)

    has_errors = any(d["status"] == "wrong" for d in details)
    return {
        "input":   text,
        "output":  " ".join(output),
        "status":  "corrected" if has_errors else "ok",
        "details": details
    }


# ── Test word-level correction ────────────────────────────────────────────
print("="*65)
print("WORD CORRECTION  —  Beam Search top-3 suggestions")
print("="*65)
print(f"  {'Wrong':<25} {'#1 Suggestion':<22} {'Correct':<22} Match")
print("-"*65)

sample  = df_pairs.sample(min(20, len(df_pairs)), random_state=42)
matches = 0
for _, row in sample.iterrows():
    suggestions = correct_word_beam(row["wrong"], s2s_model, char_tok, beam_width=5)
    best        = suggestions[0][0] if suggestions else ""
    ok          = best == row["correct"]
    matches    += int(ok)
    print(f"  {str(row['wrong']):<25} {best:<22} {str(row['correct']):<22} {'OK' if ok else ''}")

print(f"\nWord accuracy: {matches}/{len(sample)} = {matches/len(sample)*100:.1f}%")

# ── Test full sentence ────────────────────────────────────────────────────
print("\n" + "="*65)
print("FULL SENTENCE CORRECTION")
print("="*65)

test_sentences = [
    "यसरी नेपालमा वस्तु बिक्री हुने गर्न मन्त्रालयको निर्णय",
    "ीसरय नेपालमा वतु्स पिबी नेहु ्नगर ्मनलाोत्करय निर्णय",
    "अस्पतालले फोहर उपयोग गर्न अनिवार्य",
    "पतलअला्ेस होफर पयोगउ गर्न रनयाअिव्",
]

for sent in test_sentences:
    r = predict_sentence(sent, detector, s2s_model, tokenizer, char_tok)
    print(f"\nInput  : {r['input']}")
    print(f"Output : {r['output']}")
    for d in r["details"]:
        if d["status"] == "wrong":
            top3 = "  |  ".join(
                [f"{w} ({s:.2f})" for w, s in d["suggestions"]]
            )
            print(f"  {d['word']} -> [{top3}]")


## 11. Save All Models

In [ ]:
import json as _json

torch.save(detector.state_dict(),  "detector_best.pth")
torch.save(s2s_model.state_dict(), "seq2seq_best.pth")
print("Models saved.")

with open("detect_char_tokenizer.json", "w", encoding="utf-8") as f:
    _json.dump({
        "char2idx": tokenizer.char2idx,
        "idx2char": {str(k): v for k, v in tokenizer.idx2char.items()}
    }, f, ensure_ascii=False, indent=2)

with open("seq2seq_char_tokenizer.json", "w", encoding="utf-8") as f:
    _json.dump({
        "char2idx": char_tok.char2idx,
        "idx2char": {str(k): v for k, v in char_tok.idx2char.items()}
    }, f, ensure_ascii=False, indent=2)

print("Tokenizers saved.")
print("\nTo run inference on a new sentence:")
print("  result = predict_sentence(sentence, detector, s2s_model, tokenizer, char_tok)")
print("  print(result['output'])")
